In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulasi cekap Circuit penstabil dengan primitif Qiskit Aer

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
Halaman ini menunjukkan cara menggunakan primitif Qiskit Aer untuk mensimulasikan Circuit penstabil dengan cekap, termasuk yang tertakluk kepada hingar Pauli.

Circuit penstabil, yang juga dikenali sebagai Circuit Clifford, adalah kelas Circuit kuantum terhad yang penting dan boleh disimulasikan secara cekap secara klasik. Terdapat beberapa cara yang setara untuk mendefinisikan Circuit penstabil. Satu definisi ialah Circuit penstabil adalah Circuit kuantum yang terdiri sepenuhnya daripada Gate berikut:

- [CX](../api/qiskit/qiskit.circuit.library.CXGate)
- [Hadamard](../api/qiskit/qiskit.circuit.library.HGate)
- [S](../api/qiskit/qiskit.circuit.library.SGate)
- [Pengukuran](../api/qiskit/circuit#qiskit.circuit.Measure)

Perhatikan bahawa menggunakan Hadamard dan S, kita boleh membina sebarang Gate putaran Pauli ([$R_x$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXGate), [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), dan [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate)) yang mempunyai sudut dalam set ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$ (sehingga fasa global), jadi kita boleh menyertakan Gate ini dalam definisi juga.

Circuit penstabil penting dalam kajian pembetulan ralat kuantum. Kebolehan simulasi klasik mereka juga menjadikannya berguna untuk mengesahkan output komputer kuantum. Sebagai contoh, andaikan anda ingin melaksanakan Circuit kuantum yang menggunakan 100 Qubit pada komputer kuantum. Bagaimana anda tahu bahawa komputer kuantum berfungsi dengan betul? Circuit kuantum pada 100 Qubit adalah di luar kemampuan simulasi klasik secara kasar. Dengan mengubah suai Circuit anda supaya ia menjadi Circuit penstabil, anda boleh menjalankan Circuit pada komputer kuantum yang mempunyai struktur serupa dengan Circuit yang dikehendaki, tetapi yang boleh anda simulasikan pada komputer klasik. Dengan menyemak output komputer kuantum pada Circuit penstabil, anda boleh memperoleh keyakinan bahawa ia berfungsi dengan betul pada Circuit bukan penstabil juga. Lihat [*Evidence for the utility of quantum computing before fault tolerance*](https://www.nature.com/articles/s41586-023-06096-3) untuk contoh idea ini dalam praktik.

[Simulasi tepat dan berbising dengan primitif Qiskit Aer](simulate-with-qiskit-aer) menunjukkan cara menggunakan [Qiskit Aer](https://qiskit.org/ecosystem/aer/) untuk melakukan simulasi tepat dan berbising bagi Circuit kuantum generik. Pertimbangkan Circuit contoh yang digunakan dalam artikel tersebut, Circuit 8 Qubit yang dibina menggunakan [efficient_su2](../api/qiskit/qiskit.circuit.library.efficient_su2):

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-stabilizer-circuits/extracted-outputs/2d26ac3e-2a6a-4d73-900f-470200a63154-0.svg)

Menggunakan Qiskit Aer, kita dapat mensimulasikan Circuit ini dengan mudah. Namun, andaikan kita menetapkan bilangan Qubit kepada 500:

In [2]:
n_qubits = 500
circuit = efficient_su2(n_qubits)
# don't try to draw the circuit because it's too large

Memandangkan kos mensimulasikan Circuit kuantum berskala secara eksponen dengan bilangan Qubit, Circuit sebesar itu secara amnya akan melebihi kemampuan simulator berprestasi tinggi sekalipun seperti Qiskit Aer. Simulasi klasik bagi Circuit kuantum generik menjadi tidak praktikal apabila bilangan Qubit melebihi kira-kira 50 hingga 100 Qubit. Namun, perhatikan bahawa Circuit efficient_su2 diparameterkan dengan sudut pada Gate $R_y$ dan $R_z$. Jika semua sudut ini berada dalam set ${0, \frac{\pi}{2}, \pi, \frac{3\pi}{2}}$, maka Circuit tersebut adalah Circuit penstabil, dan ia boleh disimulasikan dengan cekap!

Dalam sel berikut, kita menjalankan Circuit dengan primitif Sampler yang disokong oleh simulator Circuit penstabil, menggunakan parameter yang dipilih secara rawak supaya Circuit dijamin menjadi Circuit penstabil.

In [3]:
import numpy as np
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Initialize a Sampler backed by the stabilizer circuit simulator
exact_sampler = Sampler(
    options=dict(backend_options=dict(method="stabilizer"))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(
    1, AerSimulator(method="stabilizer")
)
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params)
job = exact_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Simulator Circuit penstabil juga menyokong simulasi berbising, tetapi hanya untuk kelas model hingar yang terhad. Khususnya, sebarang hingar kuantum mesti dicirikan oleh saluran [ralat Pauli](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.pauli_error.html#qiskit_aer.noise.pauli_error). [Ralat depolarizing](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.depolarizing_error.html) termasuk dalam kategori ini, jadi ia boleh disimulasikan juga. Saluran hingar klasik seperti [ralat pembacaan](https://qiskit.org/ecosystem/aer/stubs/qiskit_aer.noise.ReadoutError.html) juga boleh disimulasikan.

Sel kod berikut menjalankan simulasi yang sama seperti sebelumnya, tetapi kali ini menentukan model hingar yang menambahkan ralat depolarizing sebanyak 2% kepada setiap Gate CX, serta ralat pembacaan yang membalikkan setiap bit yang diukur dengan kebarangkalian 5%.

In [4]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
bit_flip_prob = 0.05
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noise_model.add_all_qubit_readout_error(
    ReadoutError(
        [
            [1 - bit_flip_prob, bit_flip_prob],
            [bit_flip_prob, 1 - bit_flip_prob],
        ]
    )
)

noisy_sampler = Sampler(
    options=dict(
        backend_options=dict(method="stabilizer", noise_model=noise_model)
    )
)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

Sekarang, mari gunakan primitif Estimator yang disokong oleh simulator penstabil untuk mengira nilai jangkaan bagi boleh cerapan $ZZ \cdots Z$. Disebabkan struktur istimewa Circuit penstabil, hasilnya sangat mungkin adalah 0.

In [5]:
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)

exact_estimator = Estimator(
    options=dict(backend_options=dict(method="stabilizer")),
)
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.0

## Next steps

<Admonition type="tip" title="Recommendations">
  - To simulate circuits with Qiskit Aer, see [Exact and noisy simulation with Qiskit Aer primitives](./simulate-with-qiskit-sdk-primitives).
  - Review the [Qiskit Aer](https://qiskit.org/ecosystem/aer/) documentation.
</Admonition>